In [31]:
import pandas as pd
import os
import glob
import shutil

In [32]:
# === CONFIG ===
HD_ROOT = "/media/al.pedro.santos/Seagate/gaussian_football/data/processed"
LABELS_DIR = "/mnt/storage_C4/gaussian_football/data/labels"
LABELS_BACKUP_DIR = "/mnt/storage_C4/gaussian_football/data/labels_backup"
LABELS_GLOB = os.path.join(LABELS_DIR, "labels_*.csv")
DRY_RUN = False

In [33]:
# === BACKUP ===
if not os.path.exists(LABELS_BACKUP_DIR):
    shutil.copytree(LABELS_DIR, LABELS_BACKUP_DIR)
    print(f"Backup criado em {LABELS_BACKUP_DIR}")
else:
    print(f"Backup já existe em {LABELS_BACKUP_DIR}, nada feito")

Backup já existe em /mnt/storage_C4/gaussian_football/data/labels_backup, nada feito


In [34]:
# === FUNÇÃO ===
def fill_mel_paths(csv_path, hd_root, marker="data/processed/", dry_run=False):
    """Preenche mel_path em csv_path só com paths que existem de fato no hd_root."""
    df = pd.read_csv(csv_path)

    def relative_suffix(path):
        idx = path.find(marker)
        return path[idx + len(marker):] if idx != -1 else path.lstrip("/")

    def expected_mel_path(clip_path):
        return clip_path.replace("/video/", "/mel_spectograma/").replace(".mp4", ".npy")

    expected = df["clip_path"].apply(expected_mel_path)
    exists_on_hd = expected.apply(lambda p: os.path.exists(os.path.join(hd_root, relative_suffix(p))))

    df["mel_path"] = expected.where(exists_on_hd, other=None)

    n_with_audio = exists_on_hd.sum()
    print(f"{csv_path}: {n_with_audio}/{len(df)} clips com audio confirmado no HD")

    if not dry_run:
        df.to_csv(csv_path, index=False)

    return df

In [35]:
# === EXECUÇÃO ===
for csv_path in glob.glob(LABELS_GLOB):
    fill_mel_paths(csv_path, HD_ROOT, dry_run=DRY_RUN)

/mnt/storage_C4/gaussian_football/data/labels/labels_laliga.csv: 135781/162000 clips com audio confirmado no HD
/mnt/storage_C4/gaussian_football/data/labels/labels_bundesliga.csv: 76494/79500 clips com audio confirmado no HD
/mnt/storage_C4/gaussian_football/data/labels/labels_ligue-1.csv: 55500/55500 clips com audio confirmado no HD
/mnt/storage_C4/gaussian_football/data/labels/labels_ucl.csv: 141746/145500 clips com audio confirmado no HD
/mnt/storage_C4/gaussian_football/data/labels/labels_serie-a.csv: 142497/142500 clips com audio confirmado no HD
/mnt/storage_C4/gaussian_football/data/labels/labels_epl.csv: 130490/139500 clips com audio confirmado no HD


In [36]:
import pandas as pd

old = pd.read_csv("/mnt/storage_C4/gaussian_football/data/labels_backup/labels_epl.csv")
new = pd.read_csv("/mnt/storage_C4/gaussian_football/data/labels/labels_epl.csv")

print("backup (antigo):", old["mel_path"].notna().sum())
print("atual (novo):   ", new["mel_path"].notna().sum())

backup (antigo): 0
atual (novo):    130490


In [37]:
import glob

for csv_path in glob.glob("/mnt/storage_C4/gaussian_football/data/labels/labels_*.csv"):
    df = pd.read_csv(csv_path)
    print(csv_path, df["mel_path"].notna().sum())

/mnt/storage_C4/gaussian_football/data/labels/labels_laliga.csv 135781
/mnt/storage_C4/gaussian_football/data/labels/labels_bundesliga.csv 76494
/mnt/storage_C4/gaussian_football/data/labels/labels_ligue-1.csv 55500
/mnt/storage_C4/gaussian_football/data/labels/labels_ucl.csv 141746
/mnt/storage_C4/gaussian_football/data/labels/labels_serie-a.csv 142497
/mnt/storage_C4/gaussian_football/data/labels/labels_epl.csv 130490
